In [1]:
import pandas as pd
from Bio.SeqIO.FastaIO import SimpleFastaParser as sfp
import matplotlib.pyplot as plt
from collections import defaultdict
from bs4 import BeautifulSoup
import seaborn as sns
import pysam, os, glob, skbio, math, random, gzip
from Bio import SeqIO
from scipy import stats

In [2]:
def cmdir(path):
    if not os.path.isdir(path):
        os.mkdir(path)

def sbatch(name, cpus, cmd):
    return "sbatch -J %s -p serc -t 1- -c %d --mem %dG --wrap '%s'" %(name, cpus, cpus*8, cmd)

In [3]:
rootdir = "/scratch/groups/dekas/OC17_metagenomics_2022/"

In [4]:
udir = rootdir + "chemo/"
cmdir(udir)

### get/drep contigs across samples

In [6]:
cmdir(udir + "contigs/")

In [5]:
md = pd.read_csv(udir + "contigs/supplementary_tables_v3_S2.csv")
md["pathway"].value_counts()

3HP Bicycle          1996
CBB                  1128
3HP/4HB or DC/4HB     388
rTCA                   49
WL                      4
Name: pathway, dtype: int64

In [7]:
tmp = {record[0]: record[1] for record in \
          sfp(open(udir + "contigs/all_auto_scaffolds.fna"))}

for key, row in md.query("pathway!='single copy'").drop_duplicates(["scaffold", "pathway"]).iterrows():
    with open(udir + "contigs/" + row["scaffold"] + ".fna", "w") as out:
        out.write(">%s\n%s\n" %(row["scaffold"], tmp[row["scaffold"]])) 
        
tmp = {}

In [12]:
call = "dRep compare -g %s -p 20 -pa 0.80 -sa 0.95 --clusterAlg single %s" %(udir + "contigs/fasta/*fna", udir + "drep")
print(sbatch("drep", 20, call))

sbatch -J drep -p serc -t 1- -c 20 --mem 160G --wrap 'dRep compare -g /scratch/groups/dekas/OC17_metagenomics_2022/chemo/contigs/fasta/*fna -p 20 -pa 0.80 -sa 0.95 --clusterAlg single /scratch/groups/dekas/OC17_metagenomics_2022/chemo/drep'


In [6]:
cclust = pd.read_csv(udir + "drep/data_tables/Cdb.csv")
len(cclust["secondary_cluster"].unique())

657

### annotate proteins

In [27]:
cmdir(udir + "diamond")
cmdir(udir + "diamond/parts")

In [29]:
# make diamond db from gtdb
orfbin = open("/oak/stanford/groups/dekas/db/gtdb/release207/protein/gtdb_combined.orf2bin", "w")
fasta = open("/oak/stanford/groups/dekas/db/gtdb/release207/protein/gtdb_combined.faa", "w")

for i, proteome in enumerate(glob.glob("/oak/stanford/groups/dekas/db/gtdb/release207/protein/protein_faa_reps/*/*faa")):
    name = os.path.basename(proteome).split("_protein")[0]
    #with gzip.open(proteome, "rt") as handle:
    for record in sfp(open(proteome)):
        fasta.write(">%s\n%s\n" %(record[0].split(" ")[0], record[1]))
        orfbin.write("%s\t%s\n" %(record[0].split(" ")[0], name))
    print(f"{i} records processed", end="\r")
    
orfbin.close()
fasta.close()

In [30]:
# split and launch diamond
records = [r for r in sfp(open(udir + "filt_auto_scaffolds.faa"))]
wrapper = open(udir + "diamond/diamond.sh", "w")

n = math.ceil(len(records)/30)
for i in range(0, len(records),n):
    
    # write partial faa
    with open(udir + "diamond/parts/part" + str(int(i/n)+1) + ".faa", "w") as chunk:
        for record in records[i:i + n]:
            chunk.write(">%s\n%s\n" %(record[0], str(record[1])))
    
    # generate diamond call
    call = "diamond blastp -d /$OAK/db/gtdb/release207/protein/gtdb_combined.dmnd -q %s -o %s --threads 20 -b8 -c1" \
            %(udir + "diamond/parts/part" + str(int(i/n)+1) + ".faa",
              udir + "diamond/parts/part" + str(int(i/n)+1) + ".txt")
    cmd = sbatch("dmnd", 20, call)
    wrapper.write(cmd + "\n")

wrapper.close()

In [7]:
# concatenate + collect taxonomy
dmnd = pd.concat([skbio.io.read(item, format="blast+6", into=pd.DataFrame, default_columns=True) \
                      for item in glob.glob(udir + "diamond/parts/part*.txt")])
# compute coverage
faalens = {record[0].split(" ")[0]: len(record[1]) for record \
           in sfp(open(udir + "filt_auto_scaffolds.faa"))}
dmnd["qlen"] = dmnd["qseqid"].map(faalens)
dmnd["qcov"] = dmnd.apply(lambda x: (x["qend"]-x["qstart"])/x["qlen"], axis=1)
# choose best hits for each
dmnd = dmnd.sort_values(["bitscore", "qcov"], ascending=[False,False]).drop_duplicates("qseqid")
# filter for min cov /eval
dmnd = dmnd[(dmnd["evalue"]<1e-20) & (dmnd["qcov"]>0.70)]
dmnd.head()

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qlen,qcov
10184,OC1703_3000m_50m_sens_contig_356932_26,DALI01000037.1_6,99.5,2878.0,14.0,0.0,1.0,2878.0,23.0,2900.0,0.0,5681.0,2878,0.999653
21849,OC1703_4500m_50m_sens_contig_7201623_63,DAPJ01000022.1_48,94.2,2899.0,169.0,0.0,1.0,2899.0,1.0,2899.0,0.0,5482.0,2899,0.999655
18511,OC1703_3500m_50m_sens_contig_2002734_7,NYTG01000011.1_46,98.3,1805.0,31.0,0.0,1.0,1805.0,1.0,1805.0,0.0,3484.0,1805,0.999446
26335,OC1703_4500m_50m_sens_contig_2902701_8,PCDB01000047.1_12,95.8,1787.0,75.0,0.0,1.0,1787.0,20.0,1806.0,0.0,3431.0,1787,0.999440
19983,OC1703_4500m_50m_sens_contig_147147_2,QOPK01000028.1_10,97.1,1683.0,48.0,0.0,1.0,1683.0,123.0,1805.0,0.0,3202.0,1683,0.999406


### get taxonomy - gtdb

In [8]:
# merge that in and gtdbt taxonomy
orf2bin = pd.read_csv("/oak/stanford/groups/dekas/db/gtdb/release207/protein/gtdb_combined.orf2bin", sep="\t", header=None)
orf2bin.columns = ["orf", "bin"]
orf2bin = orf2bin[orf2bin["orf"].isin(dmnd["sseqid"].unique())]

In [10]:
gtdb = pd.read_csv("/oak/stanford/groups/dekas/db/gtdb/release207/taxonomy/gtdb_taxonomy.tsv", sep="\t", header=None)
gtdb.columns = ["bin", "taxonomy"]

In [11]:
dmnd = dmnd.merge(orf2bin, how="left", left_on="sseqid", right_on="orf").fillna("None")
dmnd = dmnd.merge(gtdb, how="left", on="bin").fillna("None")
dmnd["scaffold"] = dmnd["qseqid"].apply(lambda x: "_".join(x.split("_")[:-1]))
dmnd["phylum"] = dmnd["taxonomy"].apply(lambda x: x.split(";")[1] if x != "None" else "None")
dmnd["class"] = dmnd["taxonomy"].apply(lambda x: x.split(";")[2] if x != "None" else "None")
dmnd.head()

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qlen,qcov,orf,bin,taxonomy,scaffold,phylum,class
0,OC1703_3000m_50m_sens_contig_356932_26,DALI01000037.1_6,99.5,2878.0,14.0,0.0,1.0,2878.0,23.0,2900.0,0.0,5681.0,2878,0.999653,DALI01000037.1_6,GB_GCA_002496725.1,d__Archaea;p__Thermoplasmatota;c__Poseidoniia;...,OC1703_3000m_50m_sens_contig_356932,p__Thermoplasmatota,c__Poseidoniia
1,OC1703_4500m_50m_sens_contig_7201623_63,DAPJ01000022.1_48,94.2,2899.0,169.0,0.0,1.0,2899.0,1.0,2899.0,0.0,5482.0,2899,0.999655,DAPJ01000022.1_48,GB_GCA_002502095.1,d__Archaea;p__Thermoplasmatota;c__Poseidoniia;...,OC1703_4500m_50m_sens_contig_7201623,p__Thermoplasmatota,c__Poseidoniia
2,OC1703_3500m_50m_sens_contig_2002734_7,NYTG01000011.1_46,98.3,1805.0,31.0,0.0,1.0,1805.0,1.0,1805.0,0.0,3484.0,1805,0.999446,NYTG01000011.1_46,GB_GCA_002683395.1,d__Bacteria;p__Proteobacteria;c__Gammaproteoba...,OC1703_3500m_50m_sens_contig_2002734,p__Proteobacteria,c__Gammaproteobacteria
3,OC1703_4500m_50m_sens_contig_2902701_8,PCDB01000047.1_12,95.8,1787.0,75.0,0.0,1.0,1787.0,20.0,1806.0,0.0,3431.0,1787,0.999440,PCDB01000047.1_12,GB_GCA_002591625.1,d__Bacteria;p__Proteobacteria;c__Gammaproteoba...,OC1703_4500m_50m_sens_contig_2902701,p__Proteobacteria,c__Gammaproteobacteria
4,OC1703_4500m_50m_sens_contig_147147_2,QOPK01000028.1_10,97.1,1683.0,48.0,0.0,1.0,1683.0,123.0,1805.0,0.0,3202.0,1683,0.999406,QOPK01000028.1_10,GB_GCA_003331685.1,d__Bacteria;p__Proteobacteria;c__Gammaproteoba...,OC1703_4500m_50m_sens_contig_147147,p__Proteobacteria,c__Gammaproteobacteria


### reconcile

In [16]:
orf_counts = {}

for record in sfp(open(udir + "filt_auto_scaffolds.faa")):
    scaf = "_".join(record[0].split(" # ")[0].split("_")[:-1])
    if scaf not in orf_counts:
        orf_counts[scaf] = 1
    else: orf_counts[scaf] +=1
    
len(orf_counts.keys())

3348

In [17]:
tax_info = defaultdict(list)

for scaf in dmnd["scaffold"].unique():
    
    # phylum
    subtable = dmnd[dmnd["scaffold"]==scaf].groupby(["scaffold", "phylum"], \
        as_index=False).aggregate({"qseqid":"count"}).sort_values("qseqid", ascending=False)
    subtable["total_orfs"] = orf_counts[subtable["scaffold"].iloc[0]]
    subtable["perc_orfs"] = subtable.apply(lambda x: x["qseqid"]/x["total_orfs"], axis=1)
    sorted_table_phy = subtable.sort_values("perc_orfs", ascending=False)
    
    #class
    subtable = dmnd[dmnd["scaffold"]==scaf].groupby(["scaffold", "class"], \
        as_index=False).aggregate({"qseqid":"count"}).sort_values("qseqid", ascending=False)
    subtable["total_orfs"] = orf_counts[subtable["scaffold"].iloc[0]]
    subtable["perc_orfs"] = subtable.apply(lambda x: x["qseqid"]/x["total_orfs"], axis=1)
    sorted_table_cla = subtable.sort_values("perc_orfs", ascending=False)
    
    tax_info["scaffold"].append(sorted_table_phy["scaffold"].iloc[0])
    tax_info["prot_phylum_winner"].append(sorted_table_phy["phylum"].iloc[0])
    tax_info["prot_phylum_winner_perc"].append(sorted_table_phy["perc_orfs"].iloc[0])
    tax_info["prot_class_winner"].append(sorted_table_cla["class"].iloc[0])
    tax_info["prot_class_winner_perc"].append(sorted_table_cla["perc_orfs"].iloc[0])

tax_info_df = pd.DataFrame(tax_info)
tax_info_df.tail()

,scaffold,prot_phylum_winner,prot_phylum_winner_perc,prot_class_winner,prot_class_winner_perc
3342,OC1703_3000m_50m_sens_contig_995290,p__Actinobacteriota,1.000000,c__Actinomycetia,1.000000
3343,OC1703_3500m_50m_sens_contig_4462946,p__Firmicutes,1.000000,c__Bacilli,1.000000
3344,OC1703_4000m_50m_sens_contig_1594481,p__Actinobacteriota,0.500000,c__Actinomycetia,0.500000
3345,OC1703_3500m_50m_sens_contig_2103931,p__Actinobacteriota,0.333333,c__Actinomycetia,0.333333
3346,OC1703_3500m_50m_sens_contig_809782,p__Planctomycetota,0.333333,c__Alphaproteobacteria,0.333333


### curate

In [19]:
tmerge = tax_info_df

tmp = {}
for contig in glob.glob(udir + "contigs/*fna"):
    for record in sfp(open(contig)):
        tmp[record[0]]=len(record[1])
tmerge["scaffold_len"] = tmerge["scaffold"].map(tmp)
tmp = {}

In [20]:
cclust["scaffold"] = cclust["genome"].apply(lambda x: x.replace(".fna", ""))
tmerge = tmerge.merge(cclust[["scaffold", "primary_cluster","secondary_cluster"]], how="left").fillna("not clustered")
tmerge = tmerge.merge(md.query("pathway!='single copy'").drop_duplicates(["scaffold", "pathway"])[["scaffold", "pathway"]], how="left")
tmerge.head()

,scaffold,prot_phylum_winner,prot_phylum_winner_perc,prot_class_winner,prot_class_winner_perc,scaffold_len,primary_cluster,secondary_cluster,pathway
0,OC1703_3000m_50m_sens_contig_356932,p__Thermoplasmatota,0.948276,c__Poseidoniia,0.948276,69344,128,128_1,3HP/4HB or DC/4HB
1,OC1703_4500m_50m_sens_contig_7201623,p__Thermoplasmatota,0.987342,c__Poseidoniia,0.987342,98120,129,129_2,3HP/4HB or DC/4HB
2,OC1703_3500m_50m_sens_contig_2002734,p__Proteobacteria,0.875000,c__Gammaproteobacteria,0.875000,11936,7,7_1,3HP Bicycle
3,OC1703_4500m_50m_sens_contig_2902701,p__Proteobacteria,0.909091,c__Gammaproteobacteria,0.909091,16440,380,380_0,3HP Bicycle
4,OC1703_4500m_50m_sens_contig_147147,p__Proteobacteria,1.000000,c__Gammaproteobacteria,1.000000,5399,7,7_2,3HP Bicycle


In [22]:
# read back in previous
def reconcile_taxonomy(row):
    
    if row["manual"]!='None':
        return row["manual"]
    elif row["prot_phylum_winner"] == 'None':
        return "unclassified"
    elif row["prot_phylum_winner"] == "p__Proteobacteria":
        if row["prot_class_winner"] != "None":
            return "p__Proteobacteria;" + row["prot_class_winner"]
        else: return "p__Proteobacteria;unclassified"    
    else: return row["prot_phylum_winner"]
        
ncbi = pd.read_csv(udir + "diamond/curated_v2.tsv", sep="\t").fillna("None")
ncbi["previous_ncbi_taxonomy"] = ncbi.apply(reconcile_taxonomy, axis=1)
ncbi.head()

,scaffold,pathway,scaffold_len,primary_cluster,secondary_cluster,prot_phylum_winner,prot_phylum_winner_perc,prot_class_winner,prot_class_winner_perc,bin_classification,manual,previous_ncbi_taxonomy
0,OC1703_500m_50m_sens_contig_2251444,CBB,17268,1,1_0,Pseudomonadota,0.714286,Gammaproteobacteria,0.714286,d__Bacteria;p__Proteobacteria;c__Gammaproteoba...,None,Pseudomonadota;Gammaproteobacteria
1,OC1703_4500m_4000m_sens_contig_1073933,3HP Bicycle,3681,4,4_0,Acidobacteriota,0.250000,Alphaproteobacteria,0.250000,None,unclassified,unclassified
2,OC1703_4500m_150m_sens_contig_2105415,3HP/4HB or DC/4HB,8510,5,5_1,Euryarchaeota,1.000000,Halobacteria,1.000000,d__Archaea;p__Halobacteriota;c__Halobacteria;o...,None,Euryarchaeota
3,OC1703_4500m_500m_sens_contig_989476,3HP/4HB or DC/4HB,3837,5,5_2,Euryarchaeota,0.750000,Halobacteria,0.750000,d__Archaea;p__Halobacteriota;c__Halobacteria;o...,None,Euryarchaeota
4,OC1703_4500m_1000m_sens_contig_762673,CBB,7534,6,6_1,None,0.428571,None,0.428571,d__Bacteria;p__Chloroflexota;c__Dehalococcoidi...,Chloroflexota,Chloroflexota


In [23]:
tmerge = tmerge.merge(ncbi[["scaffold", "previous_ncbi_taxonomy"]], how="left")
tmerge[["scaffold", "pathway", "scaffold_len", "primary_cluster", "secondary_cluster", \
        "prot_phylum_winner", "prot_phylum_winner_perc", "prot_class_winner", "prot_class_winner_perc",
        "previous_ncbi_taxonomy"]].sort_values(["primary_cluster", \
        "secondary_cluster", "scaffold_len"], ascending=[True,True,False]).to_csv(udir + "diamond/revision_to_curate.csv", index=False)

In [27]:
def reconcile_taxonomy(row):
    
    if row["manual"]!='None':
        return row["manual"]
    elif row["prot_phylum_winner"] == 'None':
        return "unclassified"
    elif row["prot_phylum_winner"] == "p__Proteobacteria":
        if row["prot_class_winner"] != "None":
            return "p__Proteobacteria;" + row["prot_class_winner"]
        else: return "p__Proteobacteria;unclassified"    
    else: return row["prot_phylum_winner"]
    
curated = pd.read_csv(udir + "diamond/revision_curated.tsv", sep="\t").fillna("None")
curated["reconciled_taxonomy"] = curated.apply(reconcile_taxonomy, axis=1)
curated.head()

,scaffold,pathway,scaffold_len,primary_cluster,secondary_cluster,prot_phylum_winner,prot_phylum_winner_perc,prot_class_winner,prot_class_winner_perc,previous_ncbi_taxonomy,manual,reconciled_taxonomy
0,OC1703_500m_50m_sens_contig_2251444,CBB,17268,1,1_0,p__Proteobacteria,0.904762,c__Gammaproteobacteria,0.904762,Pseudomonadota;Gammaproteobacteria,None,p__Proteobacteria;c__Gammaproteobacteria
1,OC1703_4500m_4000m_sens_contig_1073933,3HP Bicycle,3681,4,4_0,p__Proteobacteria,0.500000,c__Alphaproteobacteria,0.500000,unclassified,unclassified,unclassified
2,OC1703_4500m_150m_sens_contig_2105415,3HP/4HB or DC/4HB,8510,5,5_1,p__Halobacteriota,1.000000,c__Halobacteria,1.000000,Euryarchaeota,None,p__Halobacteriota
3,OC1703_4500m_500m_sens_contig_989476,3HP/4HB or DC/4HB,3837,5,5_2,p__Halobacteriota,0.750000,c__Halobacteria,0.750000,Euryarchaeota,None,p__Halobacteriota
4,OC1703_4500m_1000m_sens_contig_762673,CBB,7534,6,6_1,p__Proteobacteria,1.000000,c__Gammaproteobacteria,0.857143,Chloroflexota,None,p__Proteobacteria;c__Gammaproteobacteria


In [29]:
curated.query("scaffold_len>=2500")[["scaffold",
    "reconciled_taxonomy"]].to_csv(rootdir + "chemo/revision_scaffold_taxonomy_2500bp.csv", sep=",", index=False)

In [30]:
curated.query("manual=='eukaryote'")[["scaffold", "pathway", 
    "scaffold_len", "reconciled_taxonomy"]].to_csv(rootdir + "chemo/revision_eukaryotic_scaffolds_1000bp.csv", sep=",", index=False)

## metabolism

### hydrogenase

In [6]:
cmdir(rootdir + "chemo/hyd")

In [9]:
with open(rootdir + "chemo/hyd/hyd_refs.faa", "w") as out:
    
    for key, row in pd.read_csv(rootdir + "chemo/lappan_et_al.csv").iterrows():
        if row["Gene name"] == "NiFe":
            orf_name = row["Query title (MAG gene)"].split(" ")[0]
            nife_type = row["Subject title (closest database match)"].split("-")[-1]
            nife_type_mod = "_".join(["NiFe"] + nife_type.split(" ")[2:])
            out.write(">%s\n%s\n" %("_".join([orf_name,nife_type_mod]), row["MAG protein sequence"]))
    
    for record in sfp(open(rootdir + "chemo/NiFe_hydrogenase.fasta")):
        cleaned_name = record[0].replace("[", "").replace("]", "").replace(" - ", "_").replace(" ", "_")
        out.write(">%s\n%s\n" %(cleaned_name, record[1]))
            
# set up diamond run
make = "diamond makedb --db %s --in %s --threads 4" %(rootdir + "chemo/hyd/hyd_refs.dmnd", \
                                                       rootdir + "chemo/hyd/hyd_refs.faa")
call = "diamond blastp -d %s -q %s -o %s --threads 4 -b8 -c1" %(rootdir + "chemo/hyd/hyd_refs.dmnd", \
                                                                 rootdir + "chemo/all_chemo_genomes.faa", \
                                                                 rootdir + "chemo/hyd/hyd.results")
print(call)

diamond blastp -d /scratch/groups/dekas/OC17_metagenomics_2022/chemo/hyd/hyd_refs.dmnd -q /scratch/groups/dekas/OC17_metagenomics_2022/chemo/all_chemo_genomes.faa -o /scratch/groups/dekas/OC17_metagenomics_2022/chemo/hyd/hyd.results --threads 4 -b8 -c1


In [12]:
otbd = {}
for proteome in glob.glob(rootdir + "chemo/faa/*.faa"):
    bin_name = os.path.basename(proteome).split("_prodigal")[0].replace(".fa","")
    for record in sfp(open(proteome)):
        otbd[record[0].split(" # ")[0]] = bin_name

In [22]:
hresults = skbio.io.read(rootdir + "chemo/hyd/hyd.results", format="blast+6", \
                         into=pd.DataFrame, default_columns=True)
gene_lens = {record[0].split(" # ")[0]:len(record[1]) for \
                record in sfp(open(rootdir + "chemo/all_chemo_genomes.faa"))}
hresults["qlen"] = hresults["qseqid"].map(gene_lens)
hresults["qcov"] = hresults.apply(lambda x: (x["qend"]-x["qstart"])/x["qlen"], axis=1)
hfilt = hresults[(hresults["pident"]>=50) & (hresults["qcov"]>=0.80)].sort_values("bitscore", ascending=False).drop_duplicates("qseqid")
hfilt["bin"] = hfilt["qseqid"].map(otbd)
hfilt["parsed_hyd_type"] = hfilt["sseqid"].apply(lambda x: x.split("_")[-1])
# for now, filter to just aerobic uptake hydrogenases (see lappan et al.)
#hfilt = hfilt[hfilt["parsed_hyd_type"].isin(["1d", "1l", "2a"])]
#hfilt[["bin", "qseqid", "sseqid", "parsed_hyd_type", \
#       "pident", "evalue", "qcov"]].to_csv(rootdir + "hyd/untersee_hyd.blast.tsv", sep="\t", index=False)
hfilt.to_csv(rootdir + "chemo/hyd/hyd_filtered_results.tsv", sep="\t", index=False)

### check dsr

In [18]:
from io import StringIO
from Bio import SeqIO
import requests

In [24]:
# parse sequences
sequences = defaultdict(list)

for record in sfp(open(rootdir + "chemo/sulfur_genes_06152026.fasta")):
    sequences["bin"].append(record[0].split(" # ")[1])
    sequences["gene"].append(record[0].split(" # ")[0].replace(">",""))
    sequences["ko"].append(record[0].split(" # ")[2])
    sequences["seq"].append(record[1])

seqdf = pd.DataFrame(sequences)
seqdf.head()

,bin,gene,ko,seq
0,OC2_150m_MAG_5,OC2_150m_contig_1261743_7,K17725,MNNNDESHFVFRQLFDKDTGTFTYLMFDSDTLEGLIIDPVKEQFDR...
1,OC2_150m_MAG_5,OC2_150m_contig_1781220_7,K11180,MDDLKGAGSSGDEKDTGDNKGKFLNPTPILDELEGGKWPSFITGFK...
2,OC2_150m_MAG_5,OC2_150m_contig_1781220_8,K11181,MNAPIERTWKTVESGPHTLEGTLHPVVVKNYGKWKYHKLIKPGVMV...
3,OC2_50m_MAG_157,OC2_50m_contig_3812599_5,K17725,MSLIFKQLFERESCTYTYLIADRETKEAAIIDAVDIMIERDTALLK...
4,OC2_50m_MAG_275,OC2_50m_contig_1127330_5,K17725,MSLIFKQLFERESCTYTYLIADSETKEAAIVDAVDIMIDRDTALLK...


In [25]:
# references
f = "https://static-content.springer.com/esm/art%3A10.1038%2Fismej.2014.208/"\
    "MediaObjects/41396_2015_BFismej2014208_MOESM44_ESM.txt"

metadata = defaultdict(list)
with open(rootdir + "chemo/muller_et_al_references.faa", "w") as out:
    for record in sfp(StringIO(requests.get(f).text)):
        
        metadata["seqid"].append(record[0].split("\t")[0])
        metadata["description"].append(record[0].split("\t")[1])
        metadata["accession"].append(record[0].split("\t")[2])
        metadata["category"].append(record[0].split("\t")[3])
        metadata["classification"].append(record[0].split("\t")[4])
        out.write(">%s\n%s\n" %(record[0], str(record[1]).replace("-","")))

metadf = pd.DataFrame(metadata)
metadf.head(2)

,seqid,description,accession,category,classification
0,PrbAero3,"Pyrobaculum aerophilum str. IM2, copy 1",NC_003364a,Thermophilic,Reductive archaeal type DsrAB;; Crenarchaeota;...
1,PrbAero4,"Pyrobaculum aerophilum str. IM2, copy 2",NC_003364b,Thermophilic,Reductive archaeal type DsrAB;; Crenarchaeota;...


In [26]:
with open(rootdir + "chemo/deepchemo.faa", "w") as out:
    
    for genome in seqdf.query("ko!='K17725'")["bin"].unique():
        
        dsra = seqdf[seqdf["bin"]==genome].query("ko=='K11180'")["seq"].iloc[0]
        dsrb = seqdf[seqdf["bin"]==genome].query("ko=='K11181'")["seq"].iloc[0]
        
        out.write(">%s\n%s\n" %(genome, dsra+dsrb))

In [27]:
# blast queries against ref
make = "makeblastdb -dbtype prot -in %s -out %s" %(rootdir + "chemo/muller_et_al_references.faa", rootdir + "chemo/muller_et_al_references.idx")
blast = "blastp -db %s -query %s -out %s -evalue 1e-3 -max_target_seqs 10 -num_threads 1 -sorthits 3 -outfmt 6" \
    %(rootdir + "chemo/muller_et_al_references.idx", rootdir + "chemo/deepchemo.faa", rootdir + "chemo/deepchemo.blast")
print(make)
print(blast)

makeblastdb -dbtype prot -in /scratch/groups/dekas/OC17_metagenomics_2022/chemo/muller_et_al_references.faa -out /scratch/groups/dekas/OC17_metagenomics_2022/chemo/muller_et_al_references.idx
blastp -db /scratch/groups/dekas/OC17_metagenomics_2022/chemo/muller_et_al_references.idx -query /scratch/groups/dekas/OC17_metagenomics_2022/chemo/deepchemo.faa -out /scratch/groups/dekas/OC17_metagenomics_2022/chemo/deepchemo.blast -evalue 1e-3 -max_target_seqs 10 -num_threads 1 -sorthits 3 -outfmt 6


In [28]:
bresults = skbio.io.read(rootdir + "chemo/deepchemo.blast", format="blast+6", \
                         into=pd.DataFrame, default_columns=True)
# add in qcov info + quality filt
gene_lens = {record.description.split(" ")[0]:len(record.seq) for \
                record in SeqIO.parse(open(rootdir + "chemo/deepchemo.faa"), "fasta")}
bresults["qlen"] = bresults["qseqid"].map(gene_lens)
bresults["qcov"] = bresults.apply(lambda x: (x["qend"]-x["qstart"])/x["qlen"], axis=1)
bfilt = bresults[(bresults["pident"]>=50) & (bresults["qcov"]>=0.70)]
bfilt = bfilt.sort_values("bitscore", ascending=False).drop_duplicates("qseqid")
bfilt.head()

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qlen,qcov
42,OC5_3000m_MAG_162,entry452,97.036,776.0,21.0,1.0,5.0,780.0,1.0,774.0,0.0,1586.0,781,0.992318
21,OC4_3000m_MAG_173,Z39SaS29,90.617,778.0,72.0,1.0,8.0,785.0,1.0,777.0,0.0,1498.0,785,0.989809
0,OC2_150m_MAG_5,UnSYyyyy,94.920,689.0,35.0,0.0,1.0,689.0,1.0,689.0,0.0,1374.0,782,0.879795
32,OC5_1000m_MAG_103,UncSUP05,82.258,806.0,98.0,2.0,1.0,762.0,1.0,805.0,0.0,1368.0,762,0.998688
11,OC3_3000m_MAG_18,WswMetag,75.583,815.0,174.0,2.0,3.0,792.0,3.0,817.0,0.0,1326.0,792,0.996212


In [29]:
dsr_merged = bfilt[["qseqid", "sseqid"]].merge(metadf, how="left", left_on="sseqid", right_on="seqid")
dsr_merged["classification"].value_counts()

Oxidative bacterial type DsrAB;; Proteobacteria; Gammaproteobacteria; Gammaproteobacteria    3
Oxidative bacterial type DsrAB;; Proteobacteria; Alphaproteobacteria; Alphaproteobacteria    1
Oxidative bacterial type DsrAB;; Proteobacteria; Deltaproteobacteria; SAR324 lineage         1
Name: classification, dtype: int64

### prune sdo

In [30]:
with open(rootdir + "chemo/putative_sdo.faa", "w") as out:
    for key, row in seqdf.iterrows():
        if row["ko"] == "K17725":
            out.write(">%s\n%s\n" %(row["gene"], row["seq"]))

In [31]:
# add seqs to reference alignment from Zhou personal communication
cmd = "mafft --add %s --keeplength %s > %s" \
    %(rootdir + "chemo/putative_sdo.faa", rootdir + "chemo/Sdo_proteins_and_ref.faa.mafft.1line.fasta",
      rootdir + "chemo/merged_sdo.mafft")
print(cmd)

mafft --add /scratch/groups/dekas/OC17_metagenomics_2022/chemo/putative_sdo.faa --keeplength /scratch/groups/dekas/OC17_metagenomics_2022/chemo/Sdo_proteins_and_ref.faa.mafft.1line.fasta > /scratch/groups/dekas/OC17_metagenomics_2022/chemo/merged_sdo.mafft


In [32]:
sdo_info = defaultdict(list)

for record in sfp(open(rootdir + "chemo/merged_sdo.mafft")):
    
    sdo_info["sequence"].append(record[0])
    sdo_info["deepchemo"].append(record[0] in [record[0] for record in \
        sfp(open(rootdir + "chemo/putative_sdo.faa"))])
    sdo_info["residue_1"].append(record[1][736])
    sdo_info["residue_2"].append(record[1][852])
    sdo_info["fullseq"].append(record[1].replace("-", ""))

sdo_df = pd.DataFrame(sdo_info)
sdo_df.head()

,sequence,deepchemo,residue_1,residue_2,fullseq
0,128-326_maxbin2_scaf2bin.035~~128-326_scaffold...,False,G,-,MFIKTLAVGPLETNCYLIGCEETGEGAVIDPGGDAPVILAAVEEAG...
1,128-326_maxbin2_scaf2bin.045~~128-326_scaffold...,False,N,-,MHRDEQTPYTIEALELGPMENFVYLVSDRASGRAAVVDPAWEVDRI...
2,128-326_maxbin2_scaf2bin.054~~128-326_scaffold...,False,G,-,MILEQLVVGPIQANCYILGDETTREAVVIDPGGDTPMILRALQARD...
3,128-326_maxbin2_scaf2bin.089~~128-326_scaffold...,False,G,-,MGLDVREVTVGPLEVNCYVLHDTGSGEAIVVDPGDEPERILDLLKP...
4,128-326_maxbin2_scaf2bin.117~~128-326_scaffold...,False,N,-,MTTKTHAIHALELGPMDNFVYLIQDLESNRGAIVDPAWDVQQVIKL...


In [33]:
sdo_df[(sdo_df["deepchemo"]==True)]

,sequence,deepchemo,residue_1,residue_2,fullseq
1333,OC2_150m_contig_1261743_7,True,D,N,MNNNDESHFVFRQLFDKDTGTFTYLMFDSDTLEGLIIDPVKEQFDR...
1334,OC2_50m_contig_3812599_5,True,D,N,MSLIFKQLFERESCTYTYLIADRETKEAAIIDAVDIMIERDTALLK...
1335,OC2_50m_contig_1127330_5,True,D,N,MSLIFKQLFERESCTYTYLIADSETKEAAIVDAVDIMIDRDTALLK...
1336,OC3_1000m_contig_2587489_23,True,D,N,MLFRQLFDPQSGTYSYLLADTDAGEAVLIDPVYEQARRDQALLSEL...
1337,OC3_2000m_contig_2957103_10,True,D,N,MIFRQLLDSVSCTYTYLLAGRPGGEALIIDPVIEKVDRYLELLKEL...
1338,OC3_3000m_contig_2975852_4,True,D,N,MIFRQLFDNVSSTYTYLLASRKGGEALLIDPVLENTGRYLKLLEEL...
1339,OC3_3000m_contig_2139231_233,True,D,N,MMFRQLFDHESFTYTYLLAQAPGSEALLIDPVLGKVDHYIRLLEEL...
1340,OC3_3000m_contig_3265060_4,True,D,N,MLIFRQLFDPQSSTYTYLLGDSASAKAILIDCVFEQSKRDIALLKE...
1341,OC3_500m_contig_681268_4,True,D,N,MTLNQNDYPTIGFRQLFDRETSTFTYLLWDLDTKAGVIIDPVREQF...
1342,OC4_150m_contig_2289948_7,True,D,N,MLIFRQLFDPQSSTYTYLLGDSVSAKAILIDCVFEQSKRDIALLKE...
